U prethodnom delu smo obavili Exploratory Data Analysis koji nam je ukazao na odsustvo linearne autokorelacije u okviru logaritamskih prinosa kriptovaluta. U ovom dokumentu ćemo iskoristiti nekoliko algoritama kako bismo dobili stope uspešnosti koje naša metoda predvidjanja mora nadmašiti kako bi predstavljala validan prediktor za kretanje kriptovaluta. Algoritmi u pitanju su:
- Majority Rule: posmatranje koje kretanje se najviše pojavljivalo u trening setu: ako se u toku treninga Bitcoin uglavnom kretao na gore, onda za ceo test predvidjamo da će ići na gore i obratno. Ovaj algoritam je izuzetno naivan i ne gleda nikakve zavisnosti izmedju ranijih i sadašnjih vrednosti, samim time ga koristimo kao prag uspešnosti koji treba nadmašiti.
- Logistička regresija: kao parametre koristimo istorijske podatke do 5 dana unazad. Razlog zašto koristimo LR ovde je jer, iako ne postoji direktna autokorelacija izmedju prinosa, moramo testirati pretpostavku da neka linearna kombinacija istorijskih podataka može ipak doprineti predvidjanju kretanja budućih vrednosti.
- Gradient Boosted Trees: takodje koristimo istorijske podatke kao parametre. Glavni razlog za korišćenje ovog algoritma jeste korišćenje nelinearnih veza izmedju istorijskih i sadašnjih podataka u svrhu predikcije.

Takodje je bitno naglasiti da koristimo walk-forward metodu umesto k-fold, jer k-fold omogućava da u trening set zadju podaci koji su iz budućnosti, kao i da u test set dodju podaci koji su se desili ranije u odnosu na pojedine obzervacije iz trening seta.
S obzirom na Ljung-Box test koji smo odradili u prošlom delu, očekujemo da će algoritmi imati stopu uspešnosti ~50%, što bi ukazalo na neadekvatnost ulaznih podataka, a ne na loše odabrane modele. U slučaju da u ovom delu nekako dobijemo uspešnost koja je značajno iznad 50%, to nam potencijalno ukazuje na curenje podataka više nego na stvarno pronađen signal.

Za kraj, samo povećanje stope uspešnosti u odnosu na Majority Rule ne mora da bude statistički značajno; razlika od jednog procentnog poena na ovom broju predikcija je u rangu statističkog šuma. Zbog toga ćemo na kraju ovog dela da uvedemo bootstrap intervale poverenja, koji će nam ukazati koliko povećanje uspešnosti je zapravo dokaz stvarnog poboljšanja.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from loader import load_dataset

In [4]:
FEATURES = ['r_lag_1', 'r_lag_2', 'r_lag_3', 'r_lag_4', 'r_lag_5']

In [5]:
def walk_forward(df, model, feature_cols, initial_train=500, step=50):
    X = df[feature_cols].values
    y = df['label'].values
    n = len(df)

    all_preds = []
    all_truth = []

    boundary = initial_train
    while boundary < n:
        test_end = min(boundary + step, n)

        X_train, y_train = X[:boundary], y[:boundary]
        X_test,  y_test  = X[boundary:test_end], y[boundary:test_end]
        n_test = test_end - boundary

        if model == 'majority':
            print(f"boundary={boundary}, ups={sum(y_train):.0f}, half={boundary/2:.0f}, "
                f"predict={'1' if sum(y_train) > boundary/2 else '0'}")
            # Predvidjanje test vrednosti na osnovu vrednosti koja se najviše pojavljivala u okviru trening vrednosti
            if sum(y_train) > boundary / 2:
                preds = n_test * [1]
            else:
                preds = n_test * [0]
        else:
            #Sprečavamo curenje podataka tako što radimo fitting samo na podacima za trening, dok test podatke samo transformišemo
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)
            model.fit(X_train_s, y_train)
            preds = model.predict(X_test_s)

        all_preds.extend(preds)
        all_truth.extend(y_test)

        boundary = test_end

    return np.array(all_preds), np.array(all_truth)

In [8]:
df = load_dataset("1d")

print(len(df))
print(df['date'].iloc[0], '->', df['date'].iloc[-1])

df['log_return'] = np.log(df['close']).diff()

future_ret = df['log_return'].shift(-1)
df['label'] = (future_ret > 0).astype(float)
df.loc[future_ret.isna(), 'label'] = np.nan

for k in range(1, 6):
    df[f'r_lag_{k}'] = df['log_return'].shift(k)

df = df.dropna().reset_index(drop=True)

models = {
    'majority':  'majority',
    'logistic':  LogisticRegression(max_iter=1000),
    'gbt':       GradientBoostingClassifier(),
}

print(f"\n{'model':<12}{'n_preds':<10}{'accuracy':<10}")
for name, mdl in models.items():
    preds, truth = walk_forward(df, mdl, FEATURES)
    acc = (preds == truth).mean()
    print(f"{name:<12}{len(preds):<10}{acc:.4f}")

1662
2022-01-01 00:00:00 -> 2026-07-20 00:00:00

model       n_preds   accuracy  
boundary=500, ups=236, half=250, predict=0
boundary=550, ups=263, half=275, predict=0
boundary=600, ups=284, half=300, predict=0
boundary=650, ups=306, half=325, predict=0
boundary=700, ups=337, half=350, predict=0
boundary=750, ups=364, half=375, predict=0
boundary=800, ups=396, half=400, predict=0
boundary=850, ups=422, half=425, predict=0
boundary=900, ups=443, half=450, predict=0
boundary=950, ups=467, half=475, predict=0
boundary=1000, ups=494, half=500, predict=0
boundary=1050, ups=523, half=525, predict=0
boundary=1100, ups=549, half=550, predict=0
boundary=1150, ups=571, half=575, predict=0
boundary=1200, ups=596, half=600, predict=0
boundary=1250, ups=625, half=625, predict=0
boundary=1300, ups=651, half=650, predict=1
boundary=1350, ups=676, half=675, predict=1
boundary=1400, ups=702, half=700, predict=1
boundary=1450, ups=723, half=725, predict=0
boundary=1500, ups=745, half=750, predict=0
boun